# Databricks/PySpark for Data Analysts/Engineers — Module 11: Lakehouse Architecture

This module ties together everything we've covered so far (the DataFrame API, Spark
SQL, Delta Lake) into one coherent picture: what a MODERN data architecture looks like
at a company using Databricks, and why it's called a **lakehouse**.

## Table of contents
1. [Data lake vs. data warehouse vs. lakehouse](#sec1)
2. [Medallion Architecture: the bronze/silver/gold layers](#sec2)
3. [Building a mini lakehouse on our data](#sec3)
4. [Unity Catalog: the basics](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Data lake vs. data warehouse vs. lakehouse

Historically, companies built two SEPARATE systems for storing data, each with
different trade-offs:

- **Data warehouse** (e.g. a traditional analytical database) -- structured data, with
  a schema enforced upfront, optimized for SQL and BI reporting. Strengths: fast
  queries, strong consistency guarantees (ACID). Weaknesses: expensive at very large
  scale, handles unstructured data poorly (images, logs, JSON with a variable shape),
  usually a closed ecosystem.
- **Data lake** (e.g. raw files on S3/ADLS/HDFS) -- ALL data types, cheap storage at
  large scale, flexible schema ("schema-on-read"). Weaknesses: no ACID (exactly the
  problems from Module 10 -- inconsistent writes, no schema enforcement), easily turns
  into a so-called "data swamp" without governance.
- **Lakehouse** -- Delta Lake (Module 10) ADDS to cheap, flexible data-lake storage the
  properties known from data warehouses: ACID transactions, schema enforcement, fast
  SQL queries, access control. The result: ONE storage location for BI reports, ad-hoc
  analysis, data engineering, AND machine learning (Modules 12-13) -- instead of
  maintaining separate copies of data in two different systems.

> ⚡ **Why this matters in practice, not just in theory**
>
> Before the lakehouse, a typical company copied the same data TWICE: once into a data lake (for data science/ML), once into a warehouse (for BI/reporting) -- with separate ETL pipelines to maintain, sync delays, and doubled storage costs. The lakehouse (Databricks + Delta Lake) eliminates this duplication: ONE source of truth, read by both Python/Spark notebooks and BI tools through plain SQL queries.

<a id="sec2"></a>
## 2. Medallion Architecture: the bronze/silver/gold layers

**Medallion Architecture** is the standard pattern for organizing data in a lakehouse --
data flows through THREE successive layers, each of increasingly higher quality and
increasingly ready for business analysis:

- **Bronze (raw)** -- data in EXACTLY the shape it arrived from the source (CSV files,
  API events, exports from another system). Stored in full, without cleaning -- this is
  your "source-of-truth backup", which you can always return to.
- **Silver (cleaned)** -- data after CLEANING: duplicates removed, types corrected,
  bad records filtered out, joined with other tables. Still at a detailed level
  (individual rows/events), but now trustworthy.
- **Gold (business)** -- data AGGREGATED into a shape ready for specific business
  consumers: daily sales summaries, metrics per customer/region, tables that feed BI
  dashboards directly.

> ⚠️ **Each layer is a SEPARATE table, not an intermediate step that disappears after processing**
>
> The key difference from traditional ETL: bronze/silver/gold are PERSISTENT tables, each independently readable. That means if the cleaning logic in the silver layer turns out to be wrong, you can fix it and RECOMPUTE silver from bronze -- without losing any source data, since bronze is still sitting there untouched.

<a id="sec3"></a>
## 3. Building a mini lakehouse on our data

Let's build a simplified bronze -> silver -> gold pipeline on our online store
dataset. For the clarity of this exercise, we use plain Parquet (as in Module 6) -- in a
REAL Databricks environment, each of these layers would be a Delta table (Module 10),
to get ACID, time travel, and MERGE INTO at every stage.

In [ ]:
import os

os.makedirs("lakehouse", exist_ok=True)

# --- BRONZE: data exactly as it arrived from the source, with no changes at all ---
orders.write.mode("overwrite").parquet("lakehouse/bronze_orders")
order_items.write.mode("overwrite").parquet("lakehouse/bronze_order_items")
customers.write.mode("overwrite").parquet("lakehouse/bronze_customers")

print("Bronze layer written.")


In [ ]:
# --- SILVER: cleaning -- drop cancelled orders (not useful for sales analysis),
# join with customers, add item value ---
from pyspark.sql import functions as F

bronze_orders = spark.read.parquet("lakehouse/bronze_orders")
bronze_items = spark.read.parquet("lakehouse/bronze_order_items")
bronze_customers = spark.read.parquet("lakehouse/bronze_customers")

silver_orders = (
    bronze_orders.filter(F.col("status") != "Cancelled")
    .join(bronze_customers, "customer_id")
    .join(bronze_items, "order_id")
    .withColumn(
        "item_value",
        F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")),
    )
)

silver_orders.write.mode("overwrite").parquet("lakehouse/silver_orders")
print(f"Silver layer: {silver_orders.count()} rows (after dropping cancelled orders).")


In [ ]:
# --- GOLD: a business-ready aggregate -- daily sales per city ---
silver = spark.read.parquet("lakehouse/silver_orders")

gold_sales_per_city = silver.groupBy("city", "order_date").agg(
    F.round(F.sum("item_value"), 2).alias("revenue"),
    F.countDistinct("order_id").alias("order_count"),
)

gold_sales_per_city.write.mode("overwrite").parquet("lakehouse/gold_sales_per_city")
gold_sales_per_city.orderBy(F.col("revenue").desc()).show(10)


> 🧱 **The same thing in Databricks -- with Delta Lake at every stage**
>
> In a real Databricks environment, each of the three writes above would use `.write.format("delta")` instead of `.write.parquet(...)`, and recomputing silver from bronze would often go through `MERGE INTO` (Module 10) instead of a full `overwrite` -- so only NEW/changed orders get added, instead of recomputing the entire history every time. Jobs/Workflows in Databricks (Module 14) schedule pipelines like this, e.g. hourly.

<a id="sec4"></a>
## 4. Unity Catalog: the basics

**Unity Catalog** is Databricks' metadata and permissions management system -- it
answers the question "what tables do we have, who can read/change them, and where did
the data come from". Key concepts:

- **A three-level namespace**: `catalog.schema.table` (e.g.
  `store.gold.sales_per_city`) -- an extra level (`catalog`) on top of the
  `schema.table` structure known from SQL (Module 8 of the SQL course), useful for
  separating, say, `production`/`development` environments or different teams.
- **Column- and row-level permissions** -- you can grant access to the WHOLE
  `gold.sales_per_city` table, but block access to one specific sensitive column (e.g.
  personal data) for selected users.
- **Lineage (data provenance)** -- Unity Catalog automatically tracks WHICH tables a
  given table was built from (e.g. that `gold.sales_per_city` comes from
  `silver.orders`, which comes from `bronze.orders`) -- invaluable when debugging
  "where did this weird number in the report come from".

> ⚡ **spark.catalog from Module 5 is a local, simplified equivalent**
>
> `spark.catalog.listTables()` from Module 5 shows temporary views in the CURRENT session -- they disappear once the notebook closes. Unity Catalog does the same thing, but at the level of the WHOLE organization: persistent tables visible to every authorized user, across every notebook and cluster, with a full history of permissions and access.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- the differences between a data lake, a data warehouse, and a lakehouse -- and why a lakehouse removes the need for two separate systems,
- the Medallion Architecture: bronze (raw) / silver (cleaned) / gold (business) layers,
- practically building a mini bronze -> silver -> gold pipeline on our data,
- Unity Catalog: the three-level namespace, permissions, and data lineage.

### 📝 Exercise 1: A bronze layer for employees

Write `employees` as a bronze layer (`lakehouse/bronze_employees`, Parquet, no changes at all).

<details>
<summary>Show solution</summary>

```python
employees.write.mode("overwrite").parquet("lakehouse/bronze_employees")
print("bronze_employees written.")
```

</details>

### 📝 Exercise 2: A silver layer -- cleaning products

Build a silver layer for `products`: filter out products priced below 5 (treated as bad data), write it as `lakehouse/silver_products`.

<details>
<summary>Show solution</summary>

```python
products.write.mode("overwrite").parquet("lakehouse/bronze_products")

bronze_products = spark.read.parquet("lakehouse/bronze_products")
silver_products = bronze_products.filter(F.col("price") >= 5)
silver_products.write.mode("overwrite").parquet("lakehouse/silver_products")
print(f"Silver: {silver_products.count()} of {bronze_products.count()} products.")
```

</details>

### 📝 Exercise 3: A gold layer -- revenue per category

Based on `lakehouse/silver_orders` from section 3, build a gold layer with total revenue per product `category` (requires joining with `products`).

<details>
<summary>Show solution</summary>

```python
silver = spark.read.parquet("lakehouse/silver_orders")

gold_per_category = silver.join(products, "product_id").groupBy("category").agg(
    F.round(F.sum("item_value"), 2).alias("revenue")
).orderBy(F.col("revenue").desc())

gold_per_category.write.mode("overwrite").parquet("lakehouse/gold_revenue_per_category")
gold_per_category.show()
```

</details>

### 📝 Exercise 4: Simulating a three-level namespace

Without a real Unity Catalog, simulate `catalog.schema.table` naming by writing to the folder `lakehouse/store/gold/sales_per_city` instead of a flat name -- this shows the actual IDEA of the hierarchy, even without full catalog functionality.

<details>
<summary>Show solution</summary>

```python
gold_sales_per_city = spark.read.parquet("lakehouse/gold_sales_per_city")
gold_sales_per_city.write.mode("overwrite").parquet(
    "lakehouse/store/gold/sales_per_city"
)
print("Written under a hierarchical path mimicking catalog.schema.table.")
```

</details>

> 🔥 **Challenge: a full pipeline with lineage documented in comments**
>
> Build a complete bronze -> silver -> gold pipeline for EMPLOYEE PERFORMANCE ANALYSIS: bronze (`orders`, `employees` -- raw), silver (joined, filtered down to status `'Delivered'`), gold (total order value per employee + region, sorted descending). Document in code comments the PROVENANCE of EVERY table (which layer/table it was built from) -- a manual simulation of what Unity Catalog tracks automatically.

<details>
<summary>Show solution</summary>

```python
# BRONZE: orders and employees in raw form
orders.write.mode("overwrite").parquet("lakehouse/bronze_orders_v2")
employees.write.mode("overwrite").parquet("lakehouse/bronze_employees_v2")

# SILVER: built from bronze_orders_v2 + bronze_employees_v2 + order_items
# (raw, not from the lakehouse -- in a real pipeline these would be separate bronze
# tables too)
bronze_ord = spark.read.parquet("lakehouse/bronze_orders_v2")
bronze_emp = spark.read.parquet("lakehouse/bronze_employees_v2")

silver_performance = (
    bronze_ord.filter(F.col("status") == "Delivered")
    .join(bronze_emp, "employee_id")
    .join(order_items, "order_id")
    .withColumn(
        "item_value",
        F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")),
    )
)
silver_performance.write.mode("overwrite").parquet("lakehouse/silver_employee_performance")

# GOLD: built from silver_employee_performance
silver = spark.read.parquet("lakehouse/silver_employee_performance")
gold_performance = silver.groupBy("employee_id", "region").agg(
    F.round(F.sum("item_value"), 2).alias("total_value")
).orderBy(F.col("total_value").desc())

gold_performance.write.mode("overwrite").parquet("lakehouse/gold_employee_performance")
gold_performance.show(10)
```

In a real Unity Catalog, this lineage (which table was built from which) would be visible automatically in the graphical interface as a lineage diagram -- here we have to describe it by hand in comments.

</details>

## What's next?

In **Module 12** we return to fully executable code -- we start **MLlib**, Spark's
machine learning module: `Pipeline`, `VectorAssembler`, `StringIndexer`, and our first
distributed regression models.